# Chapter 6: Limits & Evolution: The XOR Crisis, Live Python Exercises (Solutions)

**ECUE 3.1.3: Neural Communication** &bull; CESI Bachelor AI (Year 2)

Reference solutions for the Chapter 6 exercises. Try the [exercise notebook](../notebooks-instructor/Chapter6_LiveExercises.ipynb) yourself before looking in here.

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Sanity check:</b> run the cell below. If it prints a NumPy version with no error, your environment is ready. If not, go back to <code>Chapter1_Setup.ipynb</code>.
</div>

In [1]:
import numpy as np

print("NumPy version:", np.__version__)
print("Environment OK ✔")

NumPy version: 2.2.6
Environment OK ✔


---

## Reminder: the Neuron and the Learning Rule

Same building blocks as Chapters 4 and 5. Run the cell below so `mp_neuron` and `train_perceptron` are available in this notebook.

In [2]:
def mp_neuron(x, w, b):
    z = np.dot(w, x) + b       # weighted sum + bias (spatial summation)
    return 1 if z >= 0 else 0  # step function (all-or-nothing decision)


def train_perceptron(X, T, eta=1.0, n_epochs=20):
    w, b = np.zeros(X.shape[1]), 0.0
    mistakes = []
    for epoch in range(n_epochs):
        errors = 0
        for x, t in zip(X, T):
            y = mp_neuron(x, w, b)
            e = t - y
            w = w + eta * e * x
            b = b + eta * e
            errors += int(y != t)
        mistakes.append(errors)
        if errors == 0:
            break
    return w, b, mistakes

---

## Live Exercise: A NAND Neuron

XOR can be rewritten with gates we already know:

$$x_1 \text{ XOR } x_2 = (x_1 \text{ OR } x_2) \text{ AND } \text{NAND}(x_1, x_2)$$

| $x_1$ | $x_2$ | NAND |
|:---:|:---:|:---:|
| 0 | 0 | 1 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Fill in the weights and bias you found on paper for NAND, then check every row.

In [3]:
OR  = dict(w=[1, 1], b=-0.5)
AND = dict(w=[1, 1], b=-1.5)

NAND = dict(w=[-1, -1], b=1.5)   # AND upside down: inhibitory weights, positive bias

print("x1 x2 | NAND")
for x1 in (0, 1):
    for x2 in (0, 1):
        print(x1, x2, " |", mp_neuron([x1, x2], **NAND))

x1 x2 | NAND
0 0  | 1
0 1  | 1
1 0  | 1
1 1  | 0


---

## Live Coding: XOR as a Two-Layer Network

<div style="text-align: center;"><img src="../../decks/assets/ch6_xor_network.svg" width="480" alt="xor network diagram"></div>

1. Write `hidden_layer(x)`, returning the array `[h1, h2]` = `[OR(x), NAND(x)]`.
2. Write `xor_network(x)`, feeding the hidden layer's output into the `AND` neuron.
3. Run the check cell: every row must match the XOR target.

In [4]:
def hidden_layer(x):
    return np.array([mp_neuron(x, **OR), mp_neuron(x, **NAND)])


def xor_network(x):
    return mp_neuron(hidden_layer(x), **AND)

In [5]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
T = np.array([0, 1, 1, 0])

print("x1 x2 | h1 h2 | y | XOR")
for x, t in zip(X, T):
    h = hidden_layer(x)
    print(x[0], x[1], " |", h[0], " ", h[1], " |", xor_network(x), "|", t)

x1 x2 | h1 h2 | y | XOR
0 0  | 0   1  | 0 | 0
0 1  | 1   1  | 1 | 1
1 0  | 1   1  | 1 | 1
1 1  | 1   0  | 0 | 0


---

## Live Exercise: Learning on a Better Representation

In Chapter 5, `train_perceptron` never converged on XOR. Now compare two runs: on the **raw inputs** `X`, and on the **hidden representation** `H`, where each input is replaced by `[h1, h2]`.

**Before running**: how many mistakes do you expect in the last epoch of each run?

In [6]:
H = np.array([hidden_layer(x) for x in X])

print("hidden representation:", H.tolist())
print("raw inputs    -> mistakes per epoch:", train_perceptron(X, T)[2])
print("hidden layer  -> mistakes per epoch:", train_perceptron(H, T)[2])

hidden representation: [[0, 1], [1, 1], [1, 1], [1, 0]]
raw inputs    -> mistakes per epoch: [3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
hidden layer  -> mistakes per epoch: [3, 2, 3, 2, 2, 3, 0]


### Solution

- On `X`: the mistakes never reach 0, as in Chapter 5.
- On `H`: the rule converges in 7 epochs. In the hidden space, `(0,1)` and `(1,0)` both land on `[1, 1]`, and the problem becomes **linearly separable**.

Same learning rule, same neuron: the only change is the **representation** it receives.

---

## Live Exercise: XNOR With Three Neurons

XNOR is the other gate that no single neuron can compute: it fires when both inputs are **equal**, both on **or** both off.

$$x_1 \text{ XNOR } x_2 = (x_1 \text{ AND } x_2) \text{ OR } (x_1 \text{ NOR } x_2)$$

| $x_1$ | $x_2$ | XNOR |
|:---:|:---:|:---:|
| 0 | 0 | 1 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

1. Fill in the NOR weights (you drew its line on paper), then write `xnor_hidden(x)` and `xnor_network(x)`.
2. Run the check cell: every row must match the XNOR target.
3. Build `H_xnor`, the hidden representation of the 4 inputs. **Draw it on paper**: where do the green points land? Then run `train_perceptron` on it.

In [7]:
NOR = dict(w=[-1, -1], b=0.5)   # OR upside down


def xnor_hidden(x):
    return np.array([mp_neuron(x, **AND), mp_neuron(x, **NOR)])


def xnor_network(x):
    return mp_neuron(xnor_hidden(x), **OR)

In [8]:
T_xnor = np.array([1, 0, 0, 1])

print("x1 x2 | h1 h2 | y | XNOR")
for x, t in zip(X, T_xnor):
    h = xnor_hidden(x)
    print(x[0], x[1], " |", h[0], " ", h[1], " |", xnor_network(x), "|", t)

x1 x2 | h1 h2 | y | XNOR
0 0  | 0   1  | 1 | 1
0 1  | 0   0  | 0 | 0
1 0  | 0   0  | 0 | 0
1 1  | 1   0  | 1 | 1


In [9]:
H_xnor = np.array([xnor_hidden(x) for x in X])

print("hidden representation:", H_xnor.tolist())
print("raw inputs    -> mistakes per epoch:", train_perceptron(X, T_xnor)[2])
print("hidden layer  -> mistakes per epoch:", train_perceptron(H_xnor, T_xnor)[2])
print("learned output neuron:", train_perceptron(H_xnor, T_xnor)[:2])

hidden representation: [[0, 1], [0, 0], [0, 0], [1, 0]]
raw inputs    -> mistakes per epoch: [2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
hidden layer  -> mistakes per epoch: [2, 1, 2, 0]
learned output neuron: (array([1., 1.]), np.float64(-1.0))


### Solution

- Hidden space: the two **white** points `(0,1)` and `(1,0)` both land on `[0, 0]` (neither AND nor NOR fires), the green ones on `[0, 1]` and `[1, 0]`.
- One line separates them: the **OR** line. On `H_xnor`, the rule converges in 4 epochs and finds $w = (1, 1)$, $b = -1$, an OR.
- On the raw inputs, it never converges, as for XOR.

A different hidden layer than for XOR, but the same idea: the hidden neurons **move the points** until one line is enough.

---

## Chapter Bilan: When One Neuron Is Not Enough

*Synthesis exercise. No scaffolding this time.*

A downstream neuron should respond (`1`) when **exactly one** of two sensory neurons is active, and stay silent (`0`) when both are quiet or both are active. Inputs are firing rates in Hz.

| Trial | $x_1$ *(Hz)* | $x_2$ *(Hz)* | Response |
|:---:|:---:|:---:|:---:|
| 1 | 10 | 12 | 0 |
| 2 | 8  | 11 | 0 |
| 3 | 42 | 38 | 0 |
| 4 | 39 | 41 | 0 |
| 5 | 40 | 9  | 1 |
| 6 | 37 | 12 | 1 |
| 7 | 11 | 43 | 1 |
| 8 | 9  | 38 | 1 |

### Your turn

1. Train `train_perceptron` on the 8 trials. What happens to the mistakes?
2. Build a **two-layer network by hand** (same OR / NAND / AND idea), choosing hidden weights and biases that work on **firing rates**, not on 0/1.
3. Check your network on the 8 trials, then on the 4 **new** trials in `X_new`.

*Hint: look at $x_1 + x_2$ for each group of trials.*

In [10]:
X_train = np.array([[10, 12], [8, 11], [42, 38], [39, 41],
                    [40, 9], [37, 12], [11, 43], [9, 38]])
T_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_new = np.array([[12, 40], [41, 40], [9, 10], [38, 11]])
T_new = np.array([1, 0, 0, 1])

In [11]:
# 1) A single neuron cannot do it
print("single perceptron -> mistakes (last 10 epochs):", train_perceptron(X_train, T_train, n_epochs=50)[2][-10:])

# 2) Two-layer network with thresholds on the summed firing rate
H1 = dict(w=[1, 1],   b=-35)   # fires when x1 + x2 >= 35: "at least one active"
H2 = dict(w=[-1, -1], b=65)    # fires when x1 + x2 <= 65: "not both active"
OUT = dict(w=[1, 1],  b=-1.5)  # AND of the two hidden neurons


def rate_hidden(x):
    return np.array([mp_neuron(x, **H1), mp_neuron(x, **H2)])


def rate_network(x):
    return mp_neuron(rate_hidden(x), **OUT)


# 3) Check
print("sums x1 + x2:     ", X_train.sum(axis=1).tolist())
print("training trials:  ", [rate_network(x) for x in X_train], "targets:", T_train.tolist())
print("new trials:       ", [rate_network(x) for x in X_new], "targets:", T_new.tolist())

single perceptron -> mistakes (last 10 epochs): [5, 3, 6, 5, 5, 6, 5, 5, 6, 5]
sums x1 + x2:      [22, 19, 80, 80, 49, 49, 54, 47]
training trials:   [0, 0, 0, 0, 1, 1, 1, 1] targets: [0, 0, 0, 0, 1, 1, 1, 1]
new trials:        [1, 0, 0, 1] targets: [1, 0, 0, 1]


### Solution

- The single perceptron keeps making mistakes: "exactly one active" is XOR on firing rates, not linearly separable.
- The summed rate separates the three groups: about **20 Hz** (both quiet), **50 Hz** (exactly one active), **80 Hz** (both active).
- $h_1$ fires above 35 Hz, $h_2$ below 65 Hz; the output fires only when both do, i.e. in the middle band.
- Other thresholds work too: any $h_1$ bias between $-47$ and $-23$, any $h_2$ bias between $54$ and $79$.

We chose 35 and 65 by looking at the data. Choosing them **automatically** is the question of Chapter 7.

---

### Step 4: learn the last layer

Keep your hand-built hidden layer, but **learn** the output neuron: run `train_perceptron` on the hidden representation of `X_train`, then test the learned output neuron on `X_new`. Does it work? What part of the network is still hand-made?

In [12]:
H_train = np.array([rate_hidden(x) for x in X_train])
w_out, b_out, mistakes = train_perceptron(H_train, T_train)
print("learned output neuron: w =", w_out, ", b =", b_out, ", mistakes per epoch:", mistakes)
print("new trials:", [mp_neuron(rate_hidden(x), w_out, b_out) for x in X_new], "targets:", T_new.tolist())

learned output neuron: w = [2. 1.] , b = -3.0 , mistakes per epoch: [2, 3, 2, 2, 3, 2, 1, 0]
new trials: [1, 0, 0, 1] targets: [1, 0, 0, 1]


### Solution

It works: once the hidden layer has made the problem linearly separable, the perceptron rule learns the output neuron on its own. But the hidden layer is still **hand-made**: the perceptron rule has no target for $h_1$ and $h_2$. Who trains the hidden layer? That's the last question of the module, in Chapter 7.

---

## Going Further: PARITY With Three Inputs

### The gate

PARITY answers one question: is the number of active inputs **odd**?

- With 2 inputs, PARITY **is** XOR: it fires for exactly one active input.
- With 3 inputs, it is XOR chained twice: $x_1 \text{ XOR } x_2 \text{ XOR } x_3$. Think of a light with three switches: **each** active input flips the light, so the light is on after 1 or 3 flips, off after 0 or 2.

| $x_1$ | $x_2$ | $x_3$ | active inputs | PARITY |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 1 | 1 | **1** |
| 0 | 1 | 0 | 1 | **1** |
| 0 | 1 | 1 | 2 | 0 |
| 1 | 0 | 0 | 1 | **1** |
| 1 | 0 | 1 | 2 | 0 |
| 1 | 1 | 0 | 2 | 0 |
| 1 | 1 | 1 | 3 | **1** |

**Where you meet it**: the *parity bit* in computers and networks. Before sending 7 bits, the sender adds an 8th bit so that the number of 1s is even. If one bit flips on the way, the receiver computes the parity, finds it odd, and knows the message is corrupted.

**Why one neuron cannot do it**: compare MAJORITY, which also only depends on the count. MAJORITY needs **one** threshold (0 0 1 1 for 0, 1, 2, 3 active inputs). PARITY alternates (0 1 0 1): adding one active input **always** flips the answer. A weighted sum with one threshold can switch from 0 to 1 once, not back and forth. In 3D, the 8 inputs are the corners of a cube, colored like a checkerboard: no plane separates them.

### Your turn

1. Run `train_perceptron` on the 8 rows: what happens?
2. Build a hidden layer of **three neurons** with $w = (1, 1, 1)$: $h_1$ fires if **at least one** input is on, $h_2$ if **at least two**, $h_3$ if **all three**. Choose their biases.
3. Find the output neuron by hand. *Hint*: its weights are $+1$, $-1$, $+1$. What bias?
4. Run `train_perceptron` on the hidden representation. Does it find the same output neuron?

In [13]:
X3 = np.array([[0, 0, 0], [0, 0, 1], [0, 1, 0], [0, 1, 1],
               [1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]])
T3 = X3.sum(axis=1) % 2   # 1 when the number of active inputs is odd
print(T3)

[0 1 1 0 1 0 0 1]


In [14]:
# 1) one neuron
print("single perceptron -> mistakes (last 10 epochs):", train_perceptron(X3, T3, n_epochs=50)[2][-10:])

# 2) the counting hidden layer
H_COUNT = [dict(w=[1, 1, 1], b=-0.5),   # at least one
           dict(w=[1, 1, 1], b=-1.5),   # at least two
           dict(w=[1, 1, 1], b=-2.5)]   # all three


def parity_hidden(x):
    return np.array([mp_neuron(x, **n) for n in H_COUNT])


# 3) the output neuron, by hand
OUT3 = dict(w=[1, -1, 1], b=-0.5)


def parity_network(x):
    return mp_neuron(parity_hidden(x), **OUT3)


print("network:", [parity_network(x) for x in X3], "targets:", T3.tolist())

# 4) learn the output neuron
H3 = np.array([parity_hidden(x) for x in X3])
print("hidden representation:", H3.tolist())
w_out, b_out, mistakes = train_perceptron(H3, T3)
print("learned output neuron: w =", w_out, ", b =", b_out, ", mistakes per epoch:", mistakes)

single perceptron -> mistakes (last 10 epochs): [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
network: [0, 1, 1, 0, 1, 0, 0, 1] targets: [0, 1, 1, 0, 1, 0, 0, 1]
hidden representation: [[0, 0, 0], [1, 0, 0], [1, 0, 0], [1, 1, 0], [1, 0, 0], [1, 1, 0], [1, 1, 0], [1, 1, 1]]
learned output neuron: w = [ 1. -1.  1.] , b = -1.0 , mistakes per epoch: [6, 1, 0]


### Solution

- One neuron: the mistakes stay at 4 out of 8, like XOR.
- The hidden layer **counts**: with equal weights, the bias sets how many active inputs it takes (-0.5, -1.5, -2.5). The 8 inputs collapse to only 4 hidden points, one per count: `[0,0,0]`, `[1,0,0]`, `[1,1,0]`, `[1,1,1]`.
- The output adds and subtracts: $h_1 - h_2 + h_3 - 0.5$ is $-0.5, +0.5, -0.5, +0.5$ for 0, 1, 2, 3 active inputs. Odd counts fire.
- On the hidden representation, the rule converges in **3 epochs** with $w = (1, -1, 1)$, $b = -1$: the same signs as ours.

With $n$ inputs, the same recipe needs $n$ hidden neurons: one per threshold "at least $k$ inputs on".